In [1]:
import os
import sys
sys.path.append('../..')
import utils

import panel as pn
import ollama
pn.extension()

In [2]:
def get_completion_from_messages(messages, model="llama3.2", temperature=0, max_tokens=500):
    """Generate response using Ollama's LLaMA 3.2 model."""
    formatted_messages = [{"role": msg["role"], "content": msg["content"]} for msg in messages]

    try:
        response = ollama.chat(model=model, messages=formatted_messages)
        return response['message']['content'].strip()
    except Exception as e:
        print("ERROR: Ollama API call failed:", str(e))
        return "[]"

### Run through the end-to-end system to answer the user query

These helper functions are running the chain of promopts that you saw in the earlier videos.

In [3]:
customer_msg = f"""tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs or TV related products do you have?"""

products_by_category = utils.get_products_from_query(customer_msg)
category_and_product_list = utils.read_string_to_list(products_by_category)
product_info = utils.get_mentioned_product_info(category_and_product_list)
assistant_answer = utils.answer_user_msg(user_msg=customer_msg, product_info=product_info)

In [4]:
print(assistant_answer) 

I'd be happy to help you with the SmartX ProPhone and the FotoSnap DSLR Camera.

**SmartX ProPhone:**

The SmartX ProPhone is a powerful smartphone with advanced camera features. Some key details about the phone include:

* Model Number: SX-PP10
* Brand: SmartX
* Warranty: 1 year
* Rating: 4.6
* Features: 6.1-inch display, 128GB storage, 12MP dual camera, 5G connectivity
* Description: A powerful smartphone with advanced camera features.
* Price: $899.99

**FotoSnap DSLR Camera:**

The FotoSnap DSLR Camera is a versatile DSLR camera that captures stunning photos and videos. Some key details about the camera include:

* Model Number: FS-DSLR200
* Brand: FotoSnap
* Warranty: 1 year
* Rating: 4.7
* Features: 24.2MP sensor, 1080p video, 3-inch LCD, Interchangeable lenses
* Description: Capture stunning photos and videos with this versatile DSLR camera.
* Price: $599.99

As for TVs and TV-related products, we have the following options:

* **CineView 4K TV:** A stunning 4K TV with a 55-inch

### Evaluate the LLM's answer to the user with a rubric, based on the extracted product information

In [5]:
cust_prod_info = {'customer_msg': customer_msg, 'context': product_info}

In [6]:
def eval_with_rubric(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    context = test_set['context']
    completion = assistant_answer

    system_message = """
You are an evaluation assistant.

Your job is to compare a customer-service answer with the provided context.

IMPORTANT:
- Evaluate ONLY the customer's actual questions.
- Do NOT count the evaluation instructions as customer questions.
- Use the context as the source of truth.
- A fact is supported if it appears in the context.
- Do not require a comparison unless the customer explicitly asks for a comparison.
- A customer question is fully answered only when all important parts of that question are answered.
"""

    user_message = f"""
[BEGIN DATA]

CUSTOMER QUESTION:
{cust_msg}

CONTEXT:
{context}

ASSISTANT ANSWER:
{completion}

[END DATA]

Return your evaluation in exactly this format:

Based only on context: Y or N
Unsupported factual information: Y or N
Disagreement with context: Y or N
Number of customer questions: number
Question 1 fully answered: Y or N
Question 2 fully answered: Y or N
Fully answered questions: number

Rules:
1. Count ONLY the questions asked by the customer.
2. Do NOT count instructions in this evaluation prompt as questions.
3. A question is Y only if all important requested information is answered.
4. If the customer asks about multiple products, all requested products must be addressed.
5. If the customer asks for a category of products, all relevant products available in the context should be considered.
6. Do not require information that the customer did not ask for.
7. Prices, ratings, warranty, model numbers, and features are supported when they appear in the context.
"""

    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]

    response = get_completion_from_messages(messages)
    return response

In [7]:
evaluation_output = eval_with_rubric(cust_prod_info, assistant_answer)
print(evaluation_output)

Based only on context: N
Unsupported factual information: N
Disagreement with context: N
Number of customer questions: 2
Question 1 fully answered: Y
Question 2 fully answered: Y
Fully answered questions: 2


### Evaluate the LLM's answer to the user based on an "ideal" / "expert" (human generated) answer.

In [8]:
test_set_ideal = {
    'customer_msg': """\
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs or TV related products do you have?""",
    'ideal_answer':"""\
Of course!  The SmartX ProPhone is a powerful \
smartphone with advanced camera features. \
For instance, it has a 12MP dual camera. \
Other features include 5G wireless and 128GB storage. \
It also has a 6.1-inch display.  The price is $899.99.

The FotoSnap DSLR Camera is great for \
capturing stunning photos and videos. \
Some features include 1080p video, \
3-inch LCD, a 24.2MP sensor, \
and interchangeable lenses. \
The price is 599.99.

For TVs and TV related products, we offer 3 TVs \


All TVs offer HDR and Smart TV.

The CineView 4K TV has vibrant colors and smart features. \
Some of these features include a 55-inch display, \
'4K resolution. It's priced at 599.

The CineView 8K TV is a stunning 8K TV. \
Some features include a 65-inch display and \
8K resolution.  It's priced at 2999.99

The CineView OLED TV lets you experience vibrant colors. \
Some features include a 55-inch display and 4K resolution. \
It's priced at 1499.99.

We also offer 2 home theater products, both which include bluetooth.\
The SoundMax Home Theater is a powerful home theater system for \
an immmersive audio experience.
Its features include 5.1 channel, 1000W output, and wireless subwoofer.
It's priced at 399.99.

The SoundMax Soundbar is a sleek and powerful soundbar.
It's features include 2.1 channel, 300W output, and wireless subwoofer.
It's priced at 199.99

Are there any questions additional you may have about these products \
that you mentioned here?
Or may do you have other questions I can help you with?
    """
}

In [9]:
def eval_vs_ideal(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    ideal = test_set['ideal_answer']
    completion = assistant_answer

    system_message = """
    You are an evaluator comparing a submitted answer with an expert answer.

    Carefully compare the submitted answer with the expert answer.
    Ignore writing style, grammar, punctuation, and wording differences.

    Output ONLY one letter: A, B, C, D, or E.
    """

    user_message = f"""
[BEGIN DATA]

CUSTOMER QUESTION:
{cust_msg}

EXPERT ANSWER:
{ideal}

SUBMITTED ANSWER:
{completion}

[END DATA]

Determine which option best describes the submitted answer.

A = The submitted answer contains ONLY some of the information from the
    expert answer, and everything it says is consistent with the expert answer.
    It does NOT contain conflicting information.

B = The submitted answer contains ALL the important information from the
    expert answer PLUS additional information, and the additional information
    does not conflict with the expert answer.

C = The submitted answer contains essentially ALL the same important information
    as the expert answer. It does not meaningfully add or remove information.

D = The submitted answer contains information that CONTRADICTS the expert answer.

E = The submitted answer is NOT relevant to the customer's question or does
    not meaningfully answer the question.

IMPORTANT:
- Do not choose A just because the submitted answer is short.
- Do not choose C unless the submitted answer contains essentially all the
  important information in the expert answer.
- If the answer is unrelated to the question, choose E.
- Missing important information means it cannot be C.
- Output ONLY the letter.

Answer:
"""

    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]

    response = get_completion_from_messages(messages)

    return response.strip()

In [10]:
print(assistant_answer)

I'd be happy to help you with the SmartX ProPhone and the FotoSnap DSLR Camera.

**SmartX ProPhone:**

The SmartX ProPhone is a powerful smartphone with advanced camera features. Some key details about the phone include:

* Model Number: SX-PP10
* Brand: SmartX
* Warranty: 1 year
* Rating: 4.6
* Features: 6.1-inch display, 128GB storage, 12MP dual camera, 5G connectivity
* Description: A powerful smartphone with advanced camera features.
* Price: $899.99

**FotoSnap DSLR Camera:**

The FotoSnap DSLR Camera is a versatile DSLR camera that captures stunning photos and videos. Some key details about the camera include:

* Model Number: FS-DSLR200
* Brand: FotoSnap
* Warranty: 1 year
* Rating: 4.7
* Features: 24.2MP sensor, 1080p video, 3-inch LCD, Interchangeable lenses
* Description: Capture stunning photos and videos with this versatile DSLR camera.
* Price: $599.99

As for TVs and TV-related products, we have the following options:

* **CineView 4K TV:** A stunning 4K TV with a 55-inch

In [12]:
eval_vs_ideal(test_set_ideal, assistant_answer)

'B'

In [13]:
assistant_answer_2 = "life is like a box of chocolates"

In [14]:
# that Llama 3.2 sometimes chooses the wrong category.
eval_vs_ideal(test_set_ideal, assistant_answer_2)

'C'